# Lead Score Model

This is a theoretical representation of the model; the parameters I will be using are:

1.	Financial Qualification

2.	Need, Problem and Product Fit

3.	Authority and Decision Structure

4.	Timeline, Urgency and Buying State
  
5.	Engagement Behaviour

6.	Company and Market Fit

7.	Lead Source Quality

8.	Competitive Landscape

9.	Relationship and Trust Equality

10.	Strategic and Lifetime Value


Moving forward, I will introduce more variables within each parameter to capture every potential scenario in the lead conversion journey. I will then assign specific weights to these variables to calculate a precise metric for each parameter. Finally by weighting the parameters themselves, I will construct a robust, comprehensive scoring model that accounts for most of the edge cases also.

In [2]:
data = [
    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 1: BUDGET & FINANCIAL READINESS
    # This section captures everything about the prospect's money situation:
    # how much they can spend, how confirmed that budget is, and how
    # complex their purchasing process will be.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "b": 50000,
        # ── Estimated Budget (Prospect's Available Money) ──
        # What it means:
        #   This is the total amount of money the prospect has set aside
        #   (or could realistically set aside) to buy our solution.
        # How to find it:
        #   - The prospect may directly tell you their budget.
        #   - If not, estimate it from clues like:
        #       • Their company size and revenue
        #       • Past purchases of similar products
        #       • What competitors in their industry typically spend
        #       • Their department's known spending patterns
        # How to fill it in:
        #   Enter a whole number in your currency (e.g., 50000 means $50,000).
        #   This is YOUR BEST ESTIMATE of what THEY can spend — not what
        #   we want to charge them.
        # Example:
        #   If a prospect says "we have around $50K for this project," enter 50000.
        #   If they haven't said anything but their company size suggests
        #   they could afford $30K–$60K, pick a reasonable midpoint like 45000.

        "d": 45000,
        # ── Deal Value (Our Required Price) ──
        # What it means:
        #   This is the total price WE need to charge for our solution.
        #   It includes everything the customer would pay us:
        #       • Software license or subscription fees
        #       • Implementation and setup costs
        #       • Onboarding and training fees
        #       • Ongoing service or support fees
        #       • Any add-ons or extras tied to this deal
        # How to fill it in:
        #   Enter a whole number in your currency (e.g., 45000 means $45,000).
        # Why it matters:
        #   If our deal value (d) is HIGHER than their budget (b), the deal
        #   is at risk — they may not be able to afford us. If it's LOWER,
        #   we're in a comfortable position.
        # Example:
        #   If our quote for everything is $45,000, enter 45000.

        "c": 0.75,
        # ── Budget Confirmation Level ──
        # What it means:
        #   How confident are we that the budget number (b) above is real
        #   and accurate? This is NOT about whether they have enough money —
        #   it's about how RELIABLY we know the number.
        # How to fill it in:
        #   Pick a value from 0 to 1 based on the strongest evidence you have:
        #
        #   0.00 = No confirmation at all
        #           We have zero information about their budget. We're
        #           purely guessing based on company size or industry norms.
        #
        #   0.25 = Light confirmation (a range was mentioned)
        #           The prospect casually said something like "we're probably
        #           looking at $40K to $60K" but nothing formal. It's a hint,
        #           not a commitment.
        #
        #   0.50 = Verbal confirmation (spoken but not written)
        #           The prospect told us on a call or in a meeting: "Our budget
        #           is $50,000." We trust what they said, but there's nothing
        #           in writing to back it up.
        #
        #   0.75 = Written confirmation (email or documented)
        #           The prospect confirmed the budget in an email, a chat
        #           message, or a shared document. We have a paper trail,
        #           but it's not an official procurement document.
        #
        #   1.00 = Formal procurement document
        #           The budget is confirmed in an official document — a
        #           Purchase Order (PO), a signed budget approval, an RFP
        #           with a stated budget, or a formal procurement form.
        #           This is the gold standard.
        #
        # Example:
        #   If the prospect emailed saying "We've earmarked $50K for this,"
        #   enter 0.75.

        "t": 3,
        # ── Fiscal Year Alignment (Months Until Budget Expires) ──
        # What it means:
        #   How many MONTHS away is the prospect's budget availability from
        #   our current sales timeline? A deal is easiest when the prospect's
        #   budget is available RIGHT NOW in their current fiscal cycle. It
        #   gets harder when the money depends on a future budget period
        #   that hasn't started yet.
        # How to fill it in:
        #   Enter the number of months until the prospect's relevant budget
        #   cycle begins or expires.
        #       • 0 = Budget is available right now, in their current fiscal period.
        #       • 3 = Budget becomes available in 3 months (e.g., next quarter).
        #       • 6 = Budget is 6 months away (e.g., next half-year cycle).
        #       • 12+ = Budget depends on next fiscal year or later.
        #   Internally, the system converts this to a 0–1 score using the
        #   formula: 1 - (T / 12). So 0 months = score of 1.0 (perfect
        #   alignment), 6 months = 0.5, 12 months = 0.0.
        # Example:
        #   If the prospect's fiscal year resets in 3 months and they need
        #   the new budget to buy, enter 3.

        "f2": 0.7,
        # ── Funding Source Type ──
        # What it means:
        #   How secure and formally approved is the SOURCE of the prospect's
        #   money? Even if a prospect says "we have budget," the money could
        #   come from a shaky source (like a manager's discretionary fund)
        #   or a rock-solid source (like a board-approved capital project).
        # How to fill it in:
        #   Pick a value from 0 to 1:
        #
        #   0.30 = No identified funding source
        #           The prospect hasn't told us where the money would come
        #           from. They might want our product, but there's no clear
        #           pot of money assigned to it yet.
        #
        #   0.50 = Departmental discretionary budget
        #           A department head or manager can spend this from their
        #           own flexible budget without needing higher approval.
        #           It's real money, but it can be redirected at any time
        #           if priorities change.
        #
        #   0.70 = Allocated project budget
        #           The money has been specifically earmarked for a defined
        #           project (e.g., "CRM upgrade project — $50K"). It's more
        #           committed than discretionary funds, but hasn't gone
        #           through the highest level of formal approval.
        #
        #   1.00 = Board-approved capital expenditure
        #           The funding was formally approved at the executive or
        #           board level as a capital investment. This is the most
        #           secure type of funding — it's extremely unlikely to be
        #           pulled or redirected.
        #
        # Example:
        #   If the prospect says "this is part of our digital transformation
        #   project budget," enter 0.7.

        "m": 0.5,
        # ── Multi-Year / Long-Term Contract Willingness ──
        # What it means:
        #   How open is the prospect to signing a contract that lasts longer
        #   than one year? Long-term contracts give us revenue stability;
        #   short-term or month-to-month deals carry the risk that the
        #   customer leaves after the initial period.
        # How to fill it in:
        #   Pick a value from 0 to 1:
        #
        #   0.0 = Refuses any long-term commitment
        #         The prospect only wants a one-time purchase, a short pilot
        #         project, or a month-to-month subscription. They will NOT
        #         consider locking in for multiple years. High risk that
        #         they leave after the initial period.
        #
        #   0.5 = Open to it, but it depends on negotiation
        #         The prospect hasn't ruled out a multi-year deal, but they
        #         want to see: better pricing/discounts for committing longer,
        #         service-level guarantees (SLAs), or proof of value during
        #         a trial period first. This is an invitation for our sales
        #         team to negotiate and make a compelling case.
        #
        #   1.0 = Actively seeking a long-term partnership
        #         The ideal scenario. The prospect WANTS a strategic partner
        #         for 2–3+ years. They value stability, locked-in pricing,
        #         and a deep vendor relationship. This deal represents high
        #         financial predictability for us.
        #
        # Example:
        #   If the prospect said "we'd consider a 2-year deal if the pricing
        #   makes sense," enter 0.5.

        "p": 0.75,
        # ── Procurement Complexity ──
        # What it means:
        #   How difficult is the prospect's internal buying process? Some
        #   companies can buy with a credit card in minutes; others require
        #   months of legal reviews, committee approvals, and formal bidding.
        #   Higher complexity = lower chance of the deal closing smoothly.
        # How to fill it in:
        #   Pick a value from 0.5 to 1 (NOTE: higher = easier):
        #
        #   1.00 = Extremely simple (credit card / click-to-buy)
        #          The prospect can purchase using a corporate credit card
        #          or a simple online agreement. No formal bidding, no
        #          complex legal reviews, zero administrative overhead.
        #          The deal can close in days.
        #
        #   0.75 = Moderate complexity (internal reviews required)
        #          The deal requires passing an internal IT security review,
        #          signing a custom Master Services Agreement (MSA), and
        #          setting up a formal Purchase Order (PO) through their
        #          finance department. Expect a 1-to-3-month cycle before
        #          the deal closes.
        #
        #   0.50 = Very complex (formal RFP / bidding process)
        #          The prospect forces us to go through a massive,
        #          competitive Request for Proposal (RFP) process. This
        #          involves strict compliance documentation, endless legal
        #          back-and-forth, background checks, and multiple committee
        #          sign-offs. The sales cycle could take 6 to 12+ months,
        #          with a high risk of the deal stalling entirely.
        #
        # Example:
        #   If the prospect said "we'll need to run this through our IT
        #   security team and set up a PO," enter 0.75.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 2: SOLUTION FIT & NEED
    # This section captures how well our product matches what the prospect
    # actually needs: their industry, their problem, their current tools,
    # and how much customisation would be required.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "r": 0.8,
        # ── Industry-Level Match ──
        # What it means:
        #   How closely does the prospect's industry match industries where
        #   our company ALREADY has proven success? If we've sold to 50
        #   healthcare companies and this prospect is in healthcare, that's
        #   a strong match. If this prospect is in a brand-new industry
        #   we've never sold to, it's a weak match.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Completely new/unrelated industry — we have zero
        #             experience or reference customers here.
        #       0.5 = Adjacent industry — somewhat related to our core
        #             industries, but limited direct experience.
        #       0.8 = Strong match — we have multiple successful customers
        #             in this exact industry.
        #       1.0 = Perfect match — this is one of our top-performing
        #             industries with extensive case studies and references.
        # Example:
        #   If we primarily serve fintech and the prospect is a fintech
        #   company, enter 0.9 or 1.0.

        "s": 0.9,
        # ── Solution Fit (Problem–Product Alignment) ──
        # What it means:
        #   How directly does our product solve the prospect's SPECIFIC
        #   problem or use case? This isn't about whether our product is
        #   good in general — it's about whether it solves THIS customer's
        #   particular pain point.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Our product has essentially nothing to do with their
        #             problem.
        #       0.3 = Loosely related — our product touches on their area
        #             but wasn't built for this use case.
        #       0.5 = Partial fit — solves some aspects of their problem
        #             but leaves major gaps.
        #       0.7 = Good fit — solves most of their problem with minor
        #             gaps or workarounds needed.
        #       0.9 = Excellent fit — our product was practically designed
        #             for this exact scenario.
        #       1.0 = Perfect fit — addresses every aspect of their stated
        #             problem.
        # Example:
        #   If the prospect needs automated invoice processing and our
        #   product's core feature is automated invoice processing, enter
        #   0.9 or 1.0.

        "p": 0.7,
        # ── Problem Statement Clarity ──
        # What it means:
        #   How clearly can the prospect explain the problem they're trying
        #   to solve? A prospect who can't articulate their problem is
        #   harder to sell to (they may not even know what they need). A
        #   prospect with a detailed, documented problem statement is much
        #   more likely to buy.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Extremely vague — they say things like "we want to
        #             improve things" or "we're exploring options" with no
        #             specifics at all.
        #       0.3 = Somewhat vague — they have a general sense of the
        #             problem area but can't describe specific pain points,
        #             impacts, or goals.
        #       0.5 = Moderate clarity — they can describe the problem
        #             verbally in reasonable detail but haven't documented
        #             it or quantified the impact.
        #       0.7 = Good clarity — they have a clear understanding of
        #             the problem with some supporting data or examples.
        #       1.0 = Fully documented — they have a detailed, written
        #             problem statement with measurable impacts, specific
        #             requirements, and clear success criteria.
        # Example:
        #   If the prospect says "We lose about 15 hours per week on manual
        #   data entry, and here's a spreadsheet showing the errors," enter
        #   0.7 or 0.8.

        "cs": 1,
        # ── Current Solution Exists ──
        # What it means:
        #   Does the prospect currently use another product, tool, or method
        #   to handle the problem they want us to solve? This tells us if
        #   we're replacing something or filling a gap that has nothing in
        #   place today.
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0   = No current solution at all — they are not using any
        #             tool, software, or process for this. It's a greenfield
        #             opportunity.
        #       0.5 = Partial/informal solution — they use spreadsheets,
        #             manual processes, free tools, or a workaround that
        #             wasn't designed for this purpose.
        #       1   = Yes, a formal solution exists — they currently use a
        #             dedicated product or vendor for this (e.g., a competitor's
        #             software, an in-house built tool, etc.).
        # Example:
        #   If the prospect currently uses a competitor's CRM, enter 1.
        #   If they track everything in spreadsheets, enter 0.5.

        "d": 0.6,
        # ── Dissatisfaction Level with Current Solution ──
        # What it means:
        #   How unhappy is the prospect with what they currently use? High
        #   dissatisfaction means they're more motivated to switch. Low
        #   dissatisfaction means they might not see enough reason to change.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Fully satisfied — they're happy with what they have.
        #             Very hard to get them to switch.
        #       0.3 = Mildly dissatisfied — a few annoyances, but nothing
        #             urgent. They'd consider alternatives if it were easy.
        #       0.5 = Moderately dissatisfied — noticeable pain points that
        #             affect productivity or results. Actively looking for
        #             something better.
        #       0.7 = Very dissatisfied — significant frustration, frequent
        #             complaints, and/or measurable business impact.
        #       1.0 = Deeply dissatisfied — the current solution is actively
        #             hurting their business. They urgently need to replace it.
        # Note:
        #   If "cs" above is 0 (no current solution), you can set this to
        #   a moderate value like 0.5 to represent their dissatisfaction
        #   with having NO solution (i.e., the pain of doing nothing).
        # Example:
        #   If the prospect says "Our current tool crashes weekly and we've
        #   lost data twice," enter 0.8 or 0.9.

        "t": 0.85,
        # ── Technical Fit (Out-of-the-Box Coverage) ──
        # What it means:
        #   What percentage of the prospect's stated requirements can our
        #   product handle IMMEDIATELY — right out of the box — without
        #   any custom development, special configuration, or workarounds?
        # How to fill it in:
        #   Enter a number from 0 to 1 (think of it as a percentage):
        #       0.0  = Our product covers 0% of their requirements as-is.
        #       0.5  = Our product covers about 50% of their needs out of
        #              the box; the rest would need custom work.
        #       0.85 = Our product covers 85% of their requirements with
        #              standard features.
        #       1.0  = Our product covers 100% of their requirements with
        #              no customisation needed at all.
        # Example:
        #   If the prospect listed 10 requirements and our product handles
        #   8 of them natively, enter 0.8.

        "c": 0.2,
        # ── Customisation Effort Required ──
        # What it means:
        #   How much additional custom development, special configuration,
        #   or bespoke work would be needed to fully meet the prospect's
        #   requirements? This is the flip side of technical fit — it
        #   captures the EFFORT and COST of closing the gap.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Zero customisation needed — everything works out of
        #             the box.
        #       0.2 = Minor customisation — small configuration tweaks,
        #             simple API integrations, or minor UI adjustments.
        #       0.5 = Moderate customisation — requires dedicated developer
        #             time, custom integrations, or workflow modifications.
        #       0.8 = Heavy customisation — significant engineering effort,
        #             new feature development, or major architectural changes.
        #       1.0 = Essentially a custom-built solution — the product
        #             would need to be fundamentally altered.
        # Example:
        #   If we just need to build one custom API connector, enter 0.2.

        "c2": 1.0,
        # ── Compliance Fit ──
        # What it means:
        #   Does our product meet the legal, regulatory, security, and
        #   industry compliance requirements that the prospect MUST follow
        #   before they're even ALLOWED to buy or use our solution?
        #   In many industries (healthcare, finance, government), a customer
        #   may love the product and have budget — but CANNOT purchase it
        #   unless it passes mandatory compliance checks (e.g., HIPAA, SOC 2,
        #   GDPR, FedRAMP, ISO 27001, etc.).
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0.0 = Non-compliant — our product does NOT meet their
        #             mandatory regulatory requirements. This is often a
        #             deal-breaker.
        #       0.5 = Partially compliant — we meet some requirements but
        #             not all. There may be a path to compliance, but it
        #             requires additional work or certifications.
        #       1.0 = Fully compliant — our product meets ALL of their
        #             required compliance, security, and regulatory
        #             standards. No blockers.
        # Example:
        #   If the prospect requires SOC 2 Type II and GDPR compliance, and
        #   we have both certifications, enter 1.0. If we have SOC 2 but
        #   not GDPR, enter 0.5.

        "u": 4,
        # ── Number of Use Cases Articulated ──
        # What it means:
        #   How many DISTINCT use cases (specific ways they'd use our product)
        #   has the prospect described to us? More use cases = more ways
        #   they see value in our product, which indicates deeper interest
        #   and a more mature buying decision.
        # How to fill it in:
        #   Count the number of separate, specific use cases the prospect
        #   has mentioned. Enter a whole number (integer).
        #       1 = They've mentioned only one way they'd use the product.
        #       3 = They've described three distinct use cases.
        #       5+ = They see our product solving many different problems
        #            across their organisation — very strong signal.
        # What counts as a "use case":
        #   Each distinct scenario or workflow where they'd apply our product.
        #   For example, if they say:
        #     (1) "We'd use it for customer onboarding"
        #     (2) "We'd also use it for internal training"
        #     (3) "And for compliance documentation"
        #   That's 3 use cases.
        # Example:
        #   If the prospect described 4 different ways they'd use the product,
        #   enter 4.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 3: AUTHORITY & DECISION-MAKING
    # This section captures who we're talking to, how much power they have,
    # and whether we have access to the people who actually make the final
    # buying decision.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "r": 0.8,
        # ── Primary Contact's Role Level (Seniority) ──
        # What it means:
        #   How senior is the MAIN person we're communicating with inside
        #   the prospect's organisation? Higher seniority usually means:
        #     • Stronger influence over the buying decision
        #     • Better visibility into company priorities and budget
        #     • Faster access to other decision-makers
        # How to fill it in:
        #   Pick ONE of these values:
        #       0.2 = Individual contributor / end user
        #             (e.g., analyst, developer, coordinator)
        #             They'll use the product but have little buying power.
        #       0.4 = Team lead / supervisor
        #             (e.g., team lead, senior specialist)
        #             Some influence, but not a budget holder.
        #       0.6 = Manager / department head
        #             (e.g., marketing manager, IT director)
        #             Controls a team budget and can champion the purchase.
        #       0.8 = Senior director / VP
        #             (e.g., VP of Sales, Director of Engineering)
        #             Significant authority, can approve mid-size purchases.
        #       1.0 = C-level / executive
        #             (e.g., CEO, CFO, CTO, COO)
        #             Ultimate decision-making authority.
        # Example:
        #   If our main contact is a VP of Operations, enter 0.8.

        "d": 1,
        # ── Decision Involvement of Our Contact ──
        # What it means:
        #   How directly does our primary contact influence or control the
        #   final purchase decision? Even a senior person might not be
        #   involved in THIS particular buying decision.
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0   = No involvement — they cannot influence the decision
        #             at all. They're just an information-gatherer or user
        #             who was assigned to talk to vendors.
        #       0.5 = Influencer — they provide input, recommendations, or
        #             evaluations, but someone else makes the final call.
        #       1   = Decision-maker — they have direct authority to approve
        #             or reject the purchase, or they are THE person who
        #             signs off on the deal.
        # Example:
        #   If our contact is the VP who will personally approve the
        #   purchase order, enter 1.

        "n": 4,
        # ── Total Number of Stakeholders ──
        # What it means:
        #   How many people inside the prospect's organisation are involved
        #   in (or need to approve) the purchase decision? This includes
        #   decision-makers, influencers, evaluators, legal reviewers,
        #   finance approvers — anyone who has a say.
        # How to fill it in:
        #   Enter a whole number (integer).
        #   More stakeholders generally means a more complex and slower
        #   sales process.
        #       1–2 = Simple decision — one or two people decide.
        #       3–5 = Moderate complexity — a small buying committee.
        #       6+  = Complex enterprise sale — many people involved,
        #             higher risk of delays and conflicting opinions.
        # Example:
        #   If the prospect has a VP, an IT manager, a procurement officer,
        #   and a legal reviewer involved, enter 4.

        "o": 0.8,
        # ── Organisational Alignment (Cross-Department Buy-In) ──
        # What it means:
        #   Are MULTIPLE departments or teams inside the prospect's
        #   organisation aligned on the need for our solution? When several
        #   departments agree they need the product, the deal is much
        #   stronger and harder to kill internally.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Only one person sees the need — no broader support.
        #       0.3 = One department supports it, but others are unaware
        #             or indifferent.
        #       0.5 = Two departments see value, but there's no formal
        #             cross-team agreement.
        #       0.8 = Multiple departments are aligned and actively
        #             supporting the purchase.
        #       1.0 = Organisation-wide alignment — this is a company-wide
        #             initiative with top-down support.
        # Example:
        #   If both the Sales team and the Marketing team are pushing for
        #   our product, enter 0.7 or 0.8.

        "p": 0.7,
        # ── Direct Access to Senior Stakeholders ──
        # What it means:
        #   How easily and directly can OUR sales team reach the senior
        #   stakeholders (executives, decision-makers) at the prospect's
        #   company? Sometimes we're stuck talking to a junior contact who
        #   "passes messages up." Other times we're directly in meetings
        #   with the C-suite.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No access at all — we're blocked from reaching
        #             senior people. Our contact won't introduce us upward.
        #       0.3 = Indirect access only — we communicate through our
        #             contact, who relays information to leadership.
        #       0.5 = Occasional access — we've had one or two brief
        #             interactions with a senior person, but it's not
        #             regular.
        #       0.7 = Good access — we can schedule meetings with senior
        #             stakeholders when needed.
        #       1.0 = Full, direct access — we regularly communicate with
        #             the key decision-maker(s), and they're actively
        #             engaged in the evaluation.
        # Example:
        #   If we've had a couple of direct calls with the CTO and can
        #   email them directly, enter 0.7 or 0.8.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 4: TIMELINE & URGENCY
    # This section captures how quickly the prospect needs to make a
    # decision, what's driving their urgency, and what might slow things down.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "t": 45,
        # ── Days Until Decision ──
        # What it means:
        #   The estimated number of DAYS remaining before the prospect is
        #   expected to make a final purchase decision (yes or no).
        # How to fill it in:
        #   Enter a whole number (integer) representing calendar days.
        #       7  = They plan to decide within a week.
        #       30 = About a month out.
        #       45 = About six weeks.
        #       90 = About three months — a longer sales cycle.
        #       180+ = Very long cycle; the deal may stall.
        # Tip:
        #   Ask the prospect directly: "When are you looking to have a
        #   solution in place?" and work backward from their answer.
        # Example:
        #   If the prospect said "We want to make a decision by mid-July"
        #   and it's currently early June, enter 45.

        "t1": 1,
        # ── Trigger Event Happened ──
        # What it means:
        #   Has a specific event occurred that created urgency or accelerated
        #   the prospect's need to buy? Trigger events are things like:
        #     • A competitor launched a new product (competitive pressure)
        #     • Their current vendor announced end-of-life for their product
        #     • A new regulation takes effect on a specific date
        #     • They received new funding or budget approval
        #     • A key executive mandated the change
        #     • They experienced a major incident (security breach, outage)
        # How to fill it in:
        #   Pick ONE of these two values:
        #       0 = No trigger event — they're evaluating at their own pace,
        #           with no external pressure forcing a decision.
        #       1 = Yes, a trigger event happened — something specific is
        #           driving urgency beyond normal interest.
        # Example:
        #   If their current tool's vendor announced they're shutting down
        #   in 6 months, enter 1.

        "t2": 30,
        # ── Days Until Trigger Event Deadline ──
        # What it means:
        #   If a trigger event happened (t1 = 1), how many DAYS until the
        #   deadline associated with that event? This creates a hard
        #   timeline the prospect can't easily push back.
        # How to fill it in:
        #   Enter a whole number (integer) representing calendar days.
        #   If no trigger event happened (t1 = 0), you can enter any value
        #   (it won't heavily impact scoring), but a reasonable default
        #   is to match the "days until decision" (t) or enter 0.
        # Example:
        #   If a new regulation takes effect in 30 days and the prospect
        #   must have a compliant solution by then, enter 30.

        "ep": 0.8,
        # ── Evaluation Process Clarity ──
        # What it means:
        #   How structured and clear is the prospect's process for evaluating
        #   and selecting a vendor? A well-defined process (with clear steps,
        #   criteria, and a timeline) means the deal is more likely to move
        #   forward predictably. A vague process means the deal could stall
        #   or disappear without warning.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No process at all — the prospect has no defined way
        #             of evaluating solutions. They're "just looking" with
        #             no structure.
        #       0.3 = Loosely defined — they have a general idea but no
        #             formal steps, criteria, or timeline.
        #       0.5 = Somewhat structured — they've outlined some steps
        #             (e.g., "we'll do demos, then decide") but lack
        #             formal evaluation criteria.
        #       0.8 = Well structured — they have a clear evaluation
        #             process with defined stages, a decision timeline,
        #             and evaluation criteria.
        #       1.0 = Formal, documented process — complete with an RFP,
        #             scoring rubric, defined decision committee, and a
        #             published timeline.
        # Example:
        #   If the prospect shared an evaluation timeline with demo dates
        #   and a decision date, enter 0.8.

        "ns": 1,
        # ── Next Step Defined ──
        # What it means:
        #   Is there a clearly agreed-upon NEXT ACTION scheduled between
        #   our team and the prospect? This is one of the strongest
        #   indicators of deal momentum. If there's always a clear next
        #   step, the deal is alive and moving. If there's no next step,
        #   the deal may be stalling.
        # How to fill it in:
        #   Pick ONE of these two values:
        #       0 = No — there is no agreed-upon next step. We're waiting
        #           to hear back, or the conversation ended without
        #           scheduling anything.
        #       1 = Yes — there is a specific, scheduled next step (e.g.,
        #           a follow-up meeting on a set date, a demo scheduled,
        #           a proposal review call booked, a contract review
        #           session, etc.).
        # Example:
        #   If you have a demo scheduled for next Tuesday, enter 1.

        "cp": 0.3,
        # ── Competing Priorities ──
        # What it means:
        #   How much attention is the prospect dividing between THIS deal
        #   and OTHER business priorities? Even if they love our product,
        #   if they're juggling a merger, a product launch, and a
        #   restructuring, our deal might keep getting pushed down their
        #   to-do list.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = This is their TOP priority — they're focused almost
        #             entirely on solving this problem. Maximum attention
        #             and momentum.
        #       0.3 = Low distraction — this is a high priority for them,
        #             with only minor competing projects.
        #       0.5 = Moderate distraction — they have a few equally
        #             important projects, and our deal gets partial
        #             attention.
        #       0.7 = High distraction — many competing priorities, and
        #             our deal is not their top focus.
        #       1.0 = Extremely distracted — they are overwhelmed with
        #             other projects. Our deal is at serious risk of
        #             being deprioritised or forgotten.
        # Example:
        #   If the prospect mentioned they're also working on two other
        #   major initiatives but said ours is still a high priority,
        #   enter 0.3.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 5: ENGAGEMENT & ACTIVITY
    # This section tracks all the ways the prospect has interacted with us:
    # emails, calls, meetings, website visits, downloads, etc. More
    # engagement = stronger buying signals.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "n1": 10,
        # ── Email Opens ──
        # What it means:
        #   The total number of times the prospect has OPENED emails sent
        #   by our sales team. This is tracked by email tools (e.g.,
        #   HubSpot, Outreach, Salesloft). Each open counts — if they
        #   opened the same email 3 times, that counts as 3.
        # How to fill it in:
        #   Enter a whole number from your email tracking tool.
        #   If you don't track email opens, enter 0.
        # Example:
        #   If your CRM shows 10 total email opens from this prospect,
        #   enter 10.

        "n2": 5,
        # ── Email Replies ──
        # What it means:
        #   The total number of email RESPONSES the prospect has sent back
        #   to our team. Replies are a much stronger engagement signal
        #   than opens — they show the prospect is actively communicating
        #   with us.
        # How to fill it in:
        #   Enter a whole number. Count only replies FROM the prospect to
        #   our team, not our outbound emails.
        # Example:
        #   If the prospect has replied to 5 of our emails, enter 5.

        "n3": 3,
        # ── Meetings Completed ──
        # What it means:
        #   The number of scheduled meetings that were ACTUALLY COMPLETED
        #   with the prospect. This includes discovery calls, demo
        #   presentations, technical deep-dives, business reviews — any
        #   formal, scheduled meeting that both sides attended.
        # How to fill it in:
        #   Enter a whole number. Count only meetings that actually
        #   happened (not those that were scheduled but cancelled or
        #   no-showed).
        # Example:
        #   If you've had an intro call, a demo, and a technical review
        #   meeting, enter 3.

        "n4": 4,
        # ── Phone/Video Calls Completed ──
        # What it means:
        #   The number of meaningful phone calls or video calls completed
        #   with the prospect. These may overlap with meetings (n3) or
        #   be separate — for instance, a quick 10-minute phone check-in
        #   that wasn't a formal meeting. Count any substantive call where
        #   real business was discussed.
        # How to fill it in:
        #   Enter a whole number.
        # Example:
        #   If you've had 4 calls (including 2 formal demos and 2 informal
        #   check-ins), enter 4.

        "n5": 8,
        # ── Website Visits (Unique Sessions) ──
        # What it means:
        #   The number of unique website sessions by the prospect on OUR
        #   company website. This is tracked by analytics tools (e.g.,
        #   Google Analytics, HubSpot). If the prospect visited your site
        #   on Monday and again on Wednesday, that's 2 sessions.
        # How to fill it in:
        #   Enter a whole number from your website analytics or CRM.
        #   If you don't track this, enter 0.
        # Example:
        #   If your analytics show 8 website visits from this prospect's
        #   company, enter 8.

        "n6": 2,
        # ── Content Downloads ──
        # What it means:
        #   The number of downloadable resources the prospect has accessed
        #   from our website or sales materials. This includes:
        #     • Whitepapers or e-books
        #     • Case studies
        #     • Product datasheets
        #     • ROI calculators
        #     • Technical documentation
        #   Downloading content shows deeper interest than just visiting
        #   a webpage.
        # How to fill it in:
        #   Enter a whole number.
        # Example:
        #   If the prospect downloaded a case study and a product
        #   datasheet, enter 2.

        "n7": 3,
        # ── Pricing Page Visits ──
        # What it means:
        #   The number of times the prospect has visited our pricing page,
        #   packages page, or plan comparison page on our website. Pricing
        #   page visits are one of the strongest buying signals — they
        #   indicate the prospect is actively evaluating cost and thinking
        #   about purchasing.
        # How to fill it in:
        #   Enter a whole number from your website analytics.
        # Example:
        #   If the prospect visited the pricing page 3 separate times,
        #   enter 3.

        "n8": 1,
        # ── Demo/Trial Requests ──
        # What it means:
        #   The number of formal requests the prospect has made to see or
        #   test the product. This includes:
        #     • Requesting a live demo
        #     • Signing up for a free trial
        #     • Asking for a proof of concept (POC)
        #     • Requesting sandbox access
        #   This is a very high-intent action — they want hands-on
        #   experience with the product.
        # How to fill it in:
        #   Enter a whole number.
        # Example:
        #   If the prospect requested one product demo, enter 1.

        "n9": 2,
        # ── Social/Community Engagement ──
        # What it means:
        #   The number of interactions the prospect has had with our
        #   company through social media or community channels OUTSIDE
        #   of direct sales communication. This includes:
        #     • Liking, commenting, or sharing our LinkedIn posts
        #     • Engaging with us on Twitter/X
        #     • Participating in our community forum or Slack group
        #     • Attending our webinars or virtual events
        # How to fill it in:
        #   Enter a whole number.
        # Example:
        #   If the prospect commented on our LinkedIn post and attended
        #   a webinar, enter 2.

        "t": 7,
        # ── Days Since Last Meaningful Interaction ──
        # What it means:
        #   How many DAYS have passed since the last REAL, meaningful
        #   interaction with the prospect? "Meaningful" means a substantive
        #   exchange — a meeting, a detailed email reply, a phone call
        #   about the deal. An automated email open does NOT count.
        # How to fill it in:
        #   Enter a whole number (integer) representing calendar days.
        #       0–3  = Very recent — we just talked to them.
        #       7    = About a week ago.
        #       14   = Two weeks — starting to cool off.
        #       30+  = A month or more — the deal may be going cold.
        # Example:
        #   If the last real conversation was 7 days ago, enter 7.

        "v": 1.5,
        # ── Engagement Velocity ──
        # What it means:
        #   Is the prospect's engagement INCREASING or DECREASING over
        #   time? This is calculated as a ratio:
        #     (number of engagements in the LAST 14 days) ÷
        #     (number of engagements in the PRIOR 14 days)
        # How to interpret:
        #       < 1.0 = Engagement is SLOWING DOWN — they interacted less
        #               recently than before. Warning sign.
        #       = 1.0 = Engagement is STEADY — same level of activity.
        #       > 1.0 = Engagement is SPEEDING UP — they're becoming more
        #               active. Strong positive signal.
        # How to fill it in:
        #   Calculate or estimate the ratio. Enter a decimal number.
        #       0.5 = Engagement dropped by half (bad sign).
        #       1.0 = No change in engagement.
        #       1.5 = Engagement increased 50% (good sign).
        #       2.0 = Engagement doubled (excellent sign).
        # Example:
        #   If the prospect had 3 interactions in the prior 14 days and
        #   5 interactions in the last 14 days, enter 5/3 ≈ 1.67.

        "c": 4,
        # ── Channel Diversity ──
        # What it means:
        #   How many DIFFERENT communication channels is the prospect
        #   engaging with us through? Using multiple channels shows deeper
        #   and broader engagement.
        # How to count:
        #   Count each distinct channel as 1. Common channels:
        #     • Email
        #     • Phone
        #     • Video call (Zoom, Teams, etc.)
        #     • Website visits
        #     • Social media (LinkedIn, Twitter, etc.)
        #     • In-person meetings
        #     • Community forums / Slack
        #     • Events / webinars
        # How to fill it in:
        #   Enter a whole number.
        # Example:
        #   If the prospect communicates via email, has had phone calls,
        #   visits our website, and engages on LinkedIn, enter 4.

        "m": 1,
        # ── Negative Signals ──
        # What it means:
        #   The count of interactions that REDUCE our confidence or
        #   indicate resistance from the prospect. These are warning signs
        #   that the deal might be in trouble.
        # What counts as a negative signal:
        #     • Unsubscribing from our emails
        #     • Cancelling a scheduled meeting
        #     • No-showing to a meeting they confirmed
        #     • Saying "not interested" or "we're pausing this"
        #     • Significant delayed response after they committed to
        #       getting back to us by a certain date
        #     • Ghosting (going completely silent after active engagement)
        # How to fill it in:
        #   Enter a whole number. Count each negative event as 1.
        #       0 = No negative signals at all — great.
        #       1 = One minor negative event.
        #       3+ = Multiple warning signs — deal may be at risk.
        # Example:
        #   If the prospect cancelled one meeting, enter 1.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 6: IDEAL CUSTOMER PROFILE (ICP) FIT
    # This section checks how closely the prospect matches the TYPE of
    # customer your company is best at serving. Think of it as: "Does this
    # prospect LOOK like our best existing customers?"
    # ═══════════════════════════════════════════════════════════════════════
    {
        "seg": 0.9,
        # ── Segment Match ──
        # What it means:
        #   How closely does the prospect belong to the specific customer
        #   SEGMENT your company targets most successfully? A "segment"
        #   might be defined by industry vertical, company type, buyer
        #   persona, or business model (e.g., "mid-market B2B SaaS
        #   companies in North America").
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Completely outside our target segment.
        #       0.5 = Partially matches — overlaps with some aspects of
        #             our ideal segment but not all.
        #       0.9 = Very close match — fits our ideal customer profile
        #             almost perfectly.
        #       1.0 = Perfect match — this is exactly the type of customer
        #             we target and serve best.
        # Example:
        #   If our ideal customer is a mid-market SaaS company and this
        #   prospect is a mid-market SaaS company, enter 0.9 or 1.0.

        "emp": 0.8,
        # ── Employee Size Match ──
        # What it means:
        #   How closely does the prospect's number of employees match the
        #   company size that's ideal for our product? Some products work
        #   best for companies with 50–200 employees; others are built for
        #   enterprises with 10,000+.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Way too small or way too large for our product.
        #       0.5 = Somewhat outside our ideal size range, but could
        #             still work.
        #       0.8 = Close to our ideal employee size.
        #       1.0 = Perfectly within our ideal company size range.
        # Example:
        #   If our product works best for 200–1000 employee companies and
        #   the prospect has 500 employees, enter 0.9 or 1.0.

        "rev": 0.7,
        # ── Annual Revenue Match ──
        # What it means:
        #   How closely does the prospect's annual revenue match the
        #   revenue range of your ideal customer? This helps gauge whether
        #   they can afford our solution long-term and whether the deal
        #   size makes sense for our business.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Their revenue is far outside our ideal range (too
        #             small to afford us, or too large to care about us).
        #       0.5 = Somewhat outside our ideal range.
        #       0.7 = Close to our ideal revenue range.
        #       1.0 = Perfectly within our target revenue range.
        # Example:
        #   If our ideal customer has $10M–$50M revenue and this prospect
        #   has $30M, enter 0.9. If they have $5M, enter 0.6 or 0.7.

        "tech": 0.85,
        # ── Technology Stack Compatibility ──
        # What it means:
        #   How well does the prospect's EXISTING technology environment
        #   work with our product? Does our product integrate smoothly
        #   with the tools, systems, and platforms they already use?
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Completely incompatible — their tech stack doesn't
        #             work with our product at all.
        #       0.3 = Major compatibility issues — would require significant
        #             custom integration work.
        #       0.5 = Partially compatible — some integrations work, others
        #             don't.
        #       0.85 = Very compatible — most of their systems integrate
        #              smoothly with minor configuration.
        #       1.0 = Perfectly compatible — seamless integration with all
        #             their existing tools and systems.
        # Example:
        #   If the prospect uses Salesforce, AWS, and Slack, and our product
        #   integrates natively with all three, enter 0.9 or 1.0.

        "geo": 1.0,
        # ── Geographic Alignment ──
        # What it means:
        #   How well does the prospect's physical location fit within your
        #   company's active service regions? This matters for:
        #     • Legal/regulatory compliance (data residency laws)
        #     • Support coverage (time zones, language)
        #     • Sales team availability
        #     • On-site service requirements
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Completely outside our service area — we can't
        #             legally or practically serve them.
        #       0.3 = Fringe territory — we could serve them, but with
        #             significant logistical challenges.
        #       0.5 = Partially covered — some support limitations.
        #       0.8 = Well covered — within our active regions with minor
        #             constraints.
        #       1.0 = Perfect alignment — fully within our primary service
        #             region, time zone, and language coverage.
        # Example:
        #   If the prospect is based in a country where we have a local
        #   office and full support coverage, enter 1.0.

        "gro": 0.7,
        # ── Company Growth Trajectory ──
        # What it means:
        #   How fast is the prospect's company growing? A fast-growing
        #   company is more likely to expand their usage of our product
        #   over time (more seats, more features, higher tier).
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Declining — the company is shrinking (layoffs,
        #             revenue drops, market contraction).
        #       0.3 = Stagnant — flat growth, no significant changes.
        #       0.5 = Modest growth — growing slowly and steadily.
        #       0.7 = Good growth — expanding their team, revenue
        #             increasing, entering new markets.
        #       1.0 = Rapid growth — hypergrowth company, significant
        #             investment, fast hiring, rapidly expanding.
        # Example:
        #   If the prospect has been growing revenue 20% year-over-year
        #   and actively hiring, enter 0.7 or 0.8.

        "f": 0.8,
        # ── Financial Health / Credit Risk ──
        # What it means:
        #   How financially stable and low-risk is the prospect? A
        #   financially healthy company is more likely to pay on time,
        #   renew contracts, and expand. A financially struggling company
        #   might cancel, delay payments, or go bankrupt.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Severe financial distress — bankruptcy risk, unpaid
        #             debts, or known cash flow problems.
        #       0.3 = Concerning signs — recent layoffs, missed earnings,
        #             or negative press about financial health.
        #       0.5 = Average — nothing alarming, but no strong indicators
        #             of financial strength either.
        #       0.8 = Financially healthy — stable revenue, good credit,
        #             no red flags.
        #       1.0 = Excellent — well-funded, profitable, strong balance
        #             sheet, or backed by major investors.
        # Example:
        #   If the prospect is a profitable company with no known financial
        #   issues, enter 0.8.

        "d": 0.75,
        # ── Digital Readiness / Technology Adoption Readiness ──
        # What it means:
        #   How prepared is the prospect to adopt a digital tool like ours?
        #   Some companies are tech-savvy and quick to adopt new software.
        #   Others are still heavily manual, resistant to change, or lack
        #   the internal skills to implement new technology.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Very low readiness — the company is largely manual,
        #             resistant to technology, or lacks IT infrastructure.
        #       0.3 = Low readiness — they use basic tools but struggle
        #             with new technology adoption.
        #       0.5 = Moderate — they've adopted some digital tools but
        #             new implementations are slow and require heavy support.
        #       0.75 = Good readiness — tech-comfortable organisation with
        #              experience adopting similar tools.
        #       1.0 = Fully ready — digitally mature company that quickly
        #             adopts and integrates new technologies.
        # Example:
        #   If the prospect already uses modern cloud tools and has an IT
        #   team that manages implementations, enter 0.75 or 0.8.

        "l": 0.9,
        # ── Language & Culture Compatibility ──
        # What it means:
        #   How effectively can we WORK with this customer from a
        #   communication and business-culture perspective? This covers:
        #     • Shared language (or high English proficiency)
        #     • Compatible business norms and expectations
        #     • Similar communication styles
        #     • Time zone overlap for collaboration
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Major barriers — no shared language, completely
        #             different business culture, would require translators
        #             and cultural intermediaries.
        #       0.5 = Some challenges — we can communicate, but there are
        #             noticeable language or cultural friction points.
        #       0.9 = Very compatible — smooth communication, shared
        #             language, compatible work culture.
        #       1.0 = Perfect compatibility — same language, same business
        #             norms, same time zone, no friction at all.
        # Example:
        #   If the prospect speaks the same language and operates in a
        #   similar business culture to our team, enter 0.9 or 1.0.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 7: LEAD SOURCE QUALITY
    # This section captures HOW the prospect entered our pipeline and
    # the quality of the information we captured at that point. Better
    # sources = higher initial trust and intent.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "q": 0.75,
        # ── Source Channel Quality ──
        # What it means:
        #   How effective is the channel through which this lead first
        #   entered our sales pipeline? Some channels produce high-quality,
        #   high-intent leads; others produce low-quality leads that rarely
        #   convert. This is based on historical data about which channels
        #   work best for our company.
        # How to fill it in:
        #   Pick the value that matches how this lead found us (or how we
        #   found them). Listed from lowest to highest quality:
        #
        #   0.15 = Purchased list / cold outbound
        #          We bought their contact information or guessed their
        #          email. They've shown ZERO interest in our product. We're
        #          starting from zero trust and zero intent.
        #
        #   0.25 = Paid social ad (LinkedIn, Twitter/X, Facebook)
        #          They clicked an ad on social media. They fit our target
        #          audience, but they were originally on social media to
        #          scroll or network — not to solve a business problem.
        #
        #   0.35 = Paid search ad (Google Ads, Bing Ads)
        #          They typed a specific problem into a search engine and
        #          clicked our SPONSORED link. Shows active intent, but
        #          people naturally trust paid ads slightly less than
        #          organic results.
        #
        #   0.45 = Content marketing (blog, e-book, guide)
        #          They found and consumed our educational content. Trust
        #          is building, but they might be looking for free advice
        #          rather than a paid tool.
        #
        #   0.50 = Organic search
        #          They searched for a problem and clicked a NON-sponsored,
        #          organic link to our website. This shows both active
        #          intent and higher trust in our authority.
        #
        #   0.55 = Webinar attendee
        #          They committed an hour of their schedule to watch our
        #          presentation. Shows serious topic interest.
        #
        #   0.60 = Event / conference lead
        #          They interacted with us at a professional, industry-
        #          specific event — often face-to-face. Real-world
        #          interaction builds immediate trust.
        #
        #   0.70 = Free trial / freemium conversion
        #          They're already INSIDE our product. Intent is very high
        #          because they're actively testing whether our solution
        #          works for them. They just need to be convinced to pay.
        #
        #   0.75 = Partner referral
        #          A trusted business partner recommended us. The prospect
        #          transfers the trust they have in that partner directly
        #          to us.
        #
        #   0.85 = Inbound RFP (Request for Proposal)
        #          They sent us a formal RFP — meaning they have an active
        #          project, a timeline, and a budget. They are actively
        #          evaluating vendors and ready to write a check.
        #
        #   0.90 = Customer referral
        #          An existing, happy customer told this prospect to use
        #          our product. We don't need to prove credibility — the
        #          customer already did it for us. Highest-quality source.
        #
        # Example:
        #   If this lead came from a partner referral, enter 0.75.

        "p": 0.8,
        # ── Campaign / Asset Performance ──
        # What it means:
        #   How well did the specific marketing campaign or content asset
        #   (the ad, the webinar, the blog post, etc.) that generated this
        #   lead perform compared to our BEST-performing campaign ever?
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = The campaign performed terribly — very low conversion
        #             rates, poor lead quality.
        #       0.5 = Average campaign — performed normally compared to
        #             our other campaigns.
        #       0.8 = High-performing campaign — one of our better ones.
        #       1.0 = Our best-performing campaign — highest conversion
        #             rates and lead quality.
        # Tip:
        #   Your marketing team should be able to rank campaigns by
        #   conversion rate or lead-to-opportunity rate. Use that ranking
        #   to estimate this score.
        # Example:
        #   If this lead came from a webinar that had an 80% conversion
        #   rate (compared to our best at 100%), enter 0.8.

        "r": 0.7,
        # ── Data Richness at Point of Entry ──
        # What it means:
        #   How much information about the prospect did we capture at the
        #   moment they entered our pipeline? More data = better ability
        #   to qualify and personalise our outreach.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = We have almost nothing — just an email address.
        #       0.3 = Basic info — email + name.
        #       0.5 = Moderate info — email, name, company name, and
        #             maybe job title.
        #       0.7 = Good info — full profile including company, role,
        #             phone number, and some stated interest or context.
        #       1.0 = Complete profile — full contact details, company
        #             info, role, phone, stated problem/interest, budget
        #             range, and timeline. Everything we could want.
        # Example:
        #   If the prospect filled out a detailed form with their name,
        #   company, role, phone, and described their problem, enter 0.8
        #   or 0.9.

        "s": 1.0,
        # ── Inbound vs. Outbound ──
        # What it means:
        #   Did the prospect come to US (inbound), or did WE reach out to
        #   THEM (outbound)? Inbound leads generally have higher intent
        #   and trust because they initiated the relationship.
        # How to fill it in:
        #   Pick ONE of these two values:
        #       0.4 = Outbound — WE reached out to the prospect first
        #             (cold email, cold call, outbound campaign, etc.).
        #             They didn't ask to hear from us.
        #       1.0 = Inbound — THEY came to us (filled out a form,
        #             requested a demo, replied to content, called us,
        #             etc.). They initiated the contact.
        # Example:
        #   If the prospect submitted a demo request on our website,
        #   enter 1.0. If we cold-emailed them, enter 0.4.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 8: COMPETITIVE LANDSCAPE
    # This section captures who else the prospect is considering, how
    # strong our position is against competitors, and how locked in the
    # prospect is with their current vendor.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "n": 3,
        # ── Number of Competitors Being Evaluated ──
        # What it means:
        #   How many OTHER vendors or alternative solutions is the prospect
        #   ACTIVELY considering alongside us? More competitors = more
        #   risk of losing the deal.
        # How to fill it in:
        #   Enter a whole number (integer).
        #       0 = We're the ONLY vendor they're evaluating (see "k" below).
        #       1 = They're looking at us + 1 other option.
        #       3 = They're comparing us against 3 other vendors.
        #       5+ = Highly competitive evaluation — many vendors in the
        #            running.
        # Note:
        #   This counts competitors ONLY — not us. If they're evaluating
        #   4 vendors total including us, enter 3.
        # Example:
        #   If the prospect mentioned they're also evaluating Competitor A,
        #   Competitor B, and Competitor C, enter 3.

        "w": 0.6,
        # ── Historical Win Rate Against These Competitors ──
        # What it means:
        #   Based on our past deals, what is our success rate when we
        #   compete against these SAME competitors in THIS customer segment?
        #   This comes from our CRM data — looking at past deals where we
        #   went head-to-head with the same competitors.
        # How to fill it in:
        #   Enter a number from 0 to 1 (think of it as a percentage):
        #       0.0 = We almost never win against these competitors.
        #       0.3 = We occasionally win — maybe 30% of the time.
        #       0.5 = It's a coin flip — we win about half the time.
        #       0.6 = We win more often than we lose.
        #       0.8 = We usually win against these competitors.
        #       1.0 = We almost always win when competing with them.
        # Tip:
        #   Ask your sales operations team: "What's our win rate against
        #   [Competitor X] in [this segment]?"
        # Example:
        #   If we win about 60% of deals when competing against these
        #   same vendors, enter 0.6.

        "s": 0.3,
        # ── Incumbent / Switching Stickiness ──
        # What it means:
        #   How strongly is the prospect already tied to their EXISTING
        #   vendor or current solution? A deeply entrenched incumbent
        #   makes it much harder for us to win the deal, even if our
        #   product is better.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No incumbent — they don't currently use any
        #             competing solution, or their current vendor has
        #             very weak hold on them.
        #       0.3 = Weak incumbent — they use something, but they're
        #             not deeply invested or contractually bound.
        #       0.5 = Moderate — they have a current vendor with some
        #             integration and history, but switching is feasible.
        #       0.7 = Strong incumbent — deep integration, long contract,
        #             significant relationship with the current vendor.
        #       1.0 = Deeply entrenched — long-term contract, massive
        #             integration, the current vendor is embedded in their
        #             daily workflows. Extremely hard to displace.
        # Example:
        #   If the prospect has a basic subscription to a competitor with
        #   no deep integration, enter 0.2 or 0.3.

        "d": 0.8,
        # ── Differentiation Strength ──
        # What it means:
        #   How clearly and convincingly can we explain WHY our solution
        #   is meaningfully different and BETTER than the alternatives
        #   for this prospect's specific needs? It's not about being
        #   better in general — it's about being better for THIS
        #   particular customer's situation.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = We have no meaningful differentiation — our product
        #             is essentially the same as competitors for this use
        #             case.
        #       0.3 = Weak differentiation — minor differences that aren't
        #             compelling enough to sway the decision.
        #       0.5 = Moderate — we have some unique features or advantages,
        #             but competitors have their own strengths too.
        #       0.8 = Strong differentiation — we have clear, compelling
        #             advantages that matter to this specific customer.
        #       1.0 = Dominant differentiation — we offer something no
        #             competitor can match for this customer's needs.
        # Example:
        #   If our product has a unique AI feature that directly addresses
        #   the prospect's biggest pain point, enter 0.8 or 0.9.

        "c": 0.4,
        # ── Switching Cost ──
        # What it means:
        #   How difficult and costly would it be for the prospect to switch
        #   FROM their current approach TO our solution? High switching
        #   costs are a barrier to adoption, even if our product is better.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Trivial to switch — they can start using our product
        #             tomorrow with minimal effort.
        #       0.2 = Low cost — minor data migration or configuration.
        #       0.4 = Moderate — some data transfer, team retraining, and
        #             process changes needed.
        #       0.7 = High cost — major data migration, extensive
        #             retraining, workflow redesign, and downtime.
        #       1.0 = Massive switching cost — complete system overhaul,
        #             months of migration, significant business disruption.
        # Example:
        #   If switching requires migrating data from their old system
        #   and retraining 50 users, enter 0.4 or 0.5.

        "k": 0,
        # ── Sole Vendor Status ──
        # What it means:
        #   Are we the ONLY vendor the prospect is currently considering?
        #   Being the sole vendor dramatically increases our chances of
        #   winning — there's no competition.
        # How to fill it in:
        #   Pick ONE of these two values:
        #       0 = No — other vendors are also being evaluated (see "n"
        #           above for how many).
        #       1 = Yes — we are the ONLY vendor being considered. The
        #           prospect is not looking at any alternatives.
        # Example:
        #   If the prospect told us "you're the only vendor we're talking
        #   to," enter 1. Otherwise, enter 0.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 9: RELATIONSHIP & TRUST
    # This section captures the history and depth of our relationship
    # with the prospect. Existing relationships and trust make deals
    # significantly easier to close.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "pb": 1.0,
        # ── Previous Business Relationship ──
        # What it means:
        #   Has our company had any prior business relationship with this
        #   prospect? Selling to an existing or past customer is very
        #   different from selling to a brand-new prospect — there's
        #   already a foundation of trust (or distrust).
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0   = Net new — we have NEVER done business with this
        #             prospect. They are a completely new lead.
        #       0.5 = Lapsed customer or informal history — we had a
        #             relationship in the past but it ended (they churned
        #             or the project concluded), OR we've had informal
        #             interactions (events, partnerships) but no formal
        #             business.
        #       1.0 = Active or recent customer — they currently buy from
        #             us, or they were a customer recently and the
        #             relationship is still warm.
        # Example:
        #   If this prospect is a current customer looking to buy an
        #   additional product, enter 1.0.

        "rt": 18,
        # ── Relationship Tenure (Months) ──
        # What it means:
        #   How long (in months) have we had ANY kind of relationship with
        #   this prospect? This includes the time they've been a customer,
        #   a partner, or even just a known contact in our CRM.
        # How to fill it in:
        #   Enter a whole number (integer) representing months.
        #       0  = Brand new — we just connected with them.
        #       6  = We've known them about 6 months.
        #       18 = About 1.5 years of relationship history.
        #       36 = 3 years of history.
        # Note:
        #   If "pb" above is 0 (net new), enter 0 here as well.
        # Example:
        #   If this prospect has been a customer for 18 months, enter 18.

        "nps": 0.5,
        # ── Prospect Sentiment (Net Promoter-Style Score) ──
        # What it means:
        #   What is the prospect's overall feeling or sentiment toward
        #   our company? This is inspired by the Net Promoter Score (NPS)
        #   concept — are they a fan (promoter), neutral (passive), or
        #   unhappy with us (detractor)?
        # How to fill it in:
        #   Enter a number from -1 to 1:
        #       -1.0 = Detractor — they actively dislike our company.
        #              They've had bad experiences and would warn others
        #              away from us.
        #       -0.5 = Mildly negative — some dissatisfaction or
        #              unresolved issues, but not hostile.
        #        0.0 = Neutral / passive — no strong feelings either way.
        #              This is the default for brand-new leads with no
        #              prior history.
        #        0.5 = Mildly positive — they have a favorable impression
        #              of us but aren't enthusiastic advocates.
        #        1.0 = Promoter — they love us and would actively
        #              recommend us to others.
        # Note:
        #   For completely new leads with no prior interaction, default
        #   to 0.
        # Example:
        #   If the prospect is a current customer who has given us
        #   positive feedback but hasn't gone out of their way to promote
        #   us, enter 0.5.

        "es": 0.5,
        # ── Executive Sponsor Relationship ──
        # What it means:
        #   How strong is our personal relationship with a SENIOR EXECUTIVE
        #   or decision-maker inside the prospect's organisation? Having
        #   a strong executive sponsor can dramatically accelerate a deal
        #   and help overcome internal obstacles.
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0   = No relationship — we have not connected with any
        #             senior executive at the prospect's company.
        #       0.5 = Acquaintance — we've met or spoken with an executive
        #             (e.g., at a conference, on a brief intro call), but
        #             the relationship is not deep or personal.
        #       1.0 = Strong personal relationship — we have an ongoing,
        #             trusted relationship with a key decision-maker.
        #             They take our calls, advocate for us internally,
        #             and actively support the deal.
        # Example:
        #   If we've had one intro call with the VP but don't have a
        #   deep relationship yet, enter 0.5.

        "t": 0.7,
        # ── Trust Indicator Level ──
        # What it means:
        #   How much observable evidence is there that the prospect TRUSTS
        #   us enough to be open, transparent, and collaborative? Trust
        #   is shown through actions, not words. Look for:
        #     • Sharing confidential or internal information with us
        #     • Introducing us to other stakeholders
        #     • Giving us early access to requirements or RFP drafts
        #     • Being transparent about budget, timeline, and competitors
        #     • Asking for our strategic advice (not just product info)
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No trust signals — the prospect is guarded, shares
        #             minimal information, and keeps us at arm's length.
        #       0.3 = Low trust — basic professional politeness but no
        #             real openness.
        #       0.5 = Moderate trust — they share some information and
        #             engage in honest conversation, but keep certain
        #             details private.
        #       0.7 = Good trust — they've shared internal documents,
        #             introduced us to colleagues, or been transparent
        #             about their decision process.
        #       1.0 = High trust — they treat us as a trusted advisor,
        #             share confidential plans, and proactively include
        #             us in strategic discussions.
        # Example:
        #   If the prospect shared their internal evaluation criteria and
        #   introduced us to their IT director, enter 0.7.

        "rc": 0.8,
        # ── Reference Customer Availability ──
        # What it means:
        #   Can we point to a successful, existing customer that this
        #   prospect would find RELEVANT and CREDIBLE? The best references
        #   are customers the prospect personally knows, or companies in
        #   the same industry/size that the prospect would respect.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No relevant references — we have no customers in
        #             their industry, size, or geography. We can't show
        #             them a relatable success story.
        #       0.3 = Weak reference — we have a customer in a loosely
        #             related industry, but it's not a strong match.
        #       0.5 = Moderate — we have a relevant customer but they're
        #             not in the exact same situation, or the prospect
        #             doesn't know them.
        #       0.8 = Strong reference — we have a successful customer
        #             in the same industry, similar size, or the prospect
        #             might know them by reputation.
        #       1.0 = Perfect reference — we have a successful customer
        #             that the prospect personally knows or deeply
        #             respects (e.g., a peer company, a partner, or a
        #             well-known brand in their space).
        # Example:
        #   If we have a case study from a well-known company in the
        #   prospect's industry, enter 0.8.

        "n": 0,
        # ── Previous Negative Experience ──
        # What it means:
        #   Has the prospect had a BAD past experience with our company?
        #   If yes, how serious was it? Past negative experiences create
        #   trust barriers that we need to address before the deal can
        #   move forward.
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0   = No negative experience — either they're a new
        #             prospect with no history, or their past experience
        #             was positive/neutral.
        #       0.5 = Minor negative experience — they had a frustrating
        #             incident (e.g., a support issue, a billing error, a
        #             minor product bug) that was eventually resolved but
        #             left a slightly sour taste.
        #       1   = Major negative experience — they had a serious
        #             problem with our company (e.g., a failed
        #             implementation, a major outage, broken promises,
        #             or a bad product experience). This creates
        #             significant resistance to buying from us again.
        # Example:
        #   If the prospect is a current customer with no complaints,
        #   enter 0.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 10: STRATEGIC VALUE
    # This section looks at the LONG-TERM value of winning this deal —
    # not just the initial sale, but the lifetime revenue, expansion
    # potential, and strategic benefits for our company.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "ltv": 1500000,
        # ── Estimated Lifetime Value (LTV) ──
        # What it means:
        #   The total expected revenue our company will earn from this
        #   customer over the FULL DURATION of the relationship — not
        #   just the first deal, but all renewals, expansions, and
        #   upsells over the years.
        # How to calculate:
        #   LTV = (Annual contract value) × (Expected number of years
        #         they'll remain a customer) + (Expected expansion revenue)
        # How to fill it in:
        #   Enter a whole number in your currency.
        # Example:
        #   If the initial deal is $45K/year and you expect the customer
        #   to stay for 5 years with some expansion, resulting in a total
        #   of $300K over their lifetime, enter 300000. If the total
        #   expected lifetime revenue including expansions is $1.5M,
        #   enter 1500000.

        "cac": 400000,
        # ── Customer Acquisition Cost (CAC) ──
        # What it means:
        #   The total cost we're spending to acquire this customer. This
        #   includes all sales and marketing costs divided by the number
        #   of new customers acquired.
        #   Formula: (Total sales & marketing spend) ÷ (Number of new
        #            customers acquired in that period)
        # How to fill it in:
        #   Enter a whole number in your currency. If you don't know the
        #   exact number, use your company's average CAC.
        # Why it matters:
        #   Compare this to LTV. A healthy business has LTV much higher
        #   than CAC (typically 3× or more). If CAC is close to or higher
        #   than LTV, the deal may not be profitable.
        # Example:
        #   If our average cost to acquire a customer (sales team time,
        #   marketing spend, tools, etc.) is $400,000, enter 400000.

        "cs": 0.7,
        # ── Cross-Sell / Upsell Opportunity ──
        # What it means:
        #   How much opportunity exists to sell ADDITIONAL products or
        #   services to this customer AFTER the initial purchase? A
        #   customer who might buy more from us over time is worth more
        #   than one who will only ever buy one thing.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No cross-sell/upsell potential — this is likely a
        #             one-time purchase with no room for expansion.
        #       0.3 = Low potential — maybe one or two minor add-ons.
        #       0.5 = Moderate potential — a few additional products or
        #             tiers they could buy.
        #       0.7 = Good potential — multiple additional products,
        #             services, or higher tiers they'd likely need.
        #       1.0 = Massive potential — this customer could eventually
        #             buy our entire product suite and become one of our
        #             largest accounts.
        # Example:
        #   If we currently sell them a CRM and they could also buy our
        #   marketing automation and analytics tools, enter 0.7.

        "b": 0.6,
        # ── Brand Value of the Client ──
        # What it means:
        #   How valuable would it be to have this company as a customer
        #   from a REPUTATION and MARKETING perspective? Some customers
        #   are worth winning even at a discount because having their
        #   logo on your website opens doors to similar companies.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No brand value — unknown company, no marketing
        #             benefit from winning them.
        #       0.3 = Minor brand value — a decent company, but not one
        #             that would impress other prospects.
        #       0.5 = Moderate — a recognisable company in their industry,
        #             useful as a reference.
        #       0.8 = High brand value — a well-known company whose logo
        #             would significantly boost our credibility.
        #       1.0 = Flagship account — a world-famous brand that every
        #             prospect in our market would recognise and respect.
        # Example:
        #   If winning this customer would give us a credible reference
        #   in a new industry, enter 0.6 or 0.7.

        "m": 0.5,
        # ── Market Entry / Market Expansion Value ──
        # What it means:
        #   Would winning this customer help us enter or strengthen our
        #   presence in a NEW market (new industry, new geography, new
        #   company size segment)? Strategic deals that open new markets
        #   have extra long-term value beyond the immediate revenue.
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0   = No market entry value — this customer is in a market
        #             we already serve well. Winning them doesn't open
        #             any new doors.
        #       0.5 = Some market entry value — this deal could help us
        #             establish a foothold in a new segment, geography,
        #             or industry. It's not our primary market, but
        #             winning here could lead to more similar deals.
        #       1.0 = High market entry value — this is a strategic
        #             beachhead deal. Winning this customer would be our
        #             first success in an entirely new market that we
        #             want to expand into.
        # Example:
        #   If we've never sold to the healthcare industry and this
        #   prospect is a hospital, enter 0.5 or 1.0.

        "e": 0.65,
        # ── Expansion Revenue Probability ──
        # What it means:
        #   How likely is it that this customer will EXPAND their usage
        #   (and spending) after the initial purchase? Expansion can mean:
        #     • Adding more user seats
        #     • Upgrading to a higher tier
        #     • Buying additional modules or features
        #     • Rolling out to additional departments or locations
        # How to fill it in:
        #   Enter a number from 0 to 1 (think of it as a probability):
        #       0.0 = Very unlikely to expand — they'll use exactly what
        #             they buy, nothing more.
        #       0.3 = Low probability — some chance of minor expansion.
        #       0.5 = Moderate — about a 50/50 chance they'll expand.
        #       0.65 = Good probability — more likely than not that they'll
        #              grow their usage.
        #       1.0 = Almost certain to expand — strong signals that
        #             they'll scale up significantly after initial success.
        # Example:
        #   If the prospect said "we'd start with one department and
        #   potentially roll out company-wide," enter 0.65 or 0.7.

        "c": 0.2,
        # ── Churn Risk Indicator ──
        # What it means:
        #   How likely is it that this customer will STOP buying from us,
        #   cancel their subscription, or fail to renew in the future?
        #   High churn risk means the revenue may be short-lived, even
        #   if we win the deal.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Very low churn risk — this customer is likely to
        #             stay for years. Strong fit, strong need, high
        #             switching cost.
        #       0.2 = Low risk — a few minor concerns, but overall a
        #             sticky customer.
        #       0.5 = Moderate risk — some factors that could lead to
        #             cancellation (e.g., budget uncertainty, weak fit,
        #             or available alternatives).
        #       0.7 = High risk — significant concerns about long-term
        #             retention.
        #       1.0 = Very high risk — strong likelihood they'll cancel
        #             within the first year (e.g., they only need a
        #             one-time solution, they're financially unstable, or
        #             our product is a poor long-term fit).
        # Example:
        #   If the customer has a strong need and high switching costs
        #   (making it hard for them to leave), enter 0.1 or 0.2.
    }
]


gates = ["f", "f", "f", "f", "f", "f", "f"]
modifiers = [1, 1, 0.9, 1, 1]

weights = [0.15, 0.12, 0.20, 0.12, 0.12, 0.08, 0.04, 0.05, 0.05, 0.07]


## Financial qualification

Variables using:

B= Estimated budget(prospect’s money available), it is the total monetary amount the prospect has allocated—or is realistically able to allocate—for purchasing our solution. This may be directly stated by the prospect or estimated from available information such as past purchases, company size, department spending, or comparable deal.

D= Deal Value(our required price), it is the total quoted or expected monetary value of the solution being sold. This includes license cost, implementation, onboarding, service fees, subscription fees, or any additional charges tied to the deal.

Br(budget_ratio) = min(B/D,1) 

In ideal case our br value should tend to 1.


C= Budget confirmation level, it represents how confidently the budget amount is verified. It measures the reliability of the budget information—from unknown to fully documented, it’s a real number between 0 to 1

 0: no confirmation
 0.25: a range of budget was mentioned (light confirmation)
 0.5: verbal confirmation (no written confirmation)
 0.75: confirmation in written/through mails
 1: formal procurement document

F1= Fiscal year alignment, measures whether the customer’s budget availability matches our expected sales timeline, a deal is easier when budget is available within the current fiscal cycle and harder when funding depends on future budgets, it defines exactly how far out the prospect's buying cycle sits relative to your active fiscal calendar, it’s a real number between 0 to 1 
    = 1-(T/12)
After a year it will turn 0 

F2= funding source type, it represents how secure and formally approved the source of funding is, it’s a real number between 0 to 1. 
Rough idea for it:

0.3 = no identified source

0.5 = departmental discretionary

0.7 = allocated project budget

1.0 = board-approved capital expenditure

M= Multi-year or long-term contract willingness type, it measures the prospect’s openness toward signing a contract longer than one year, it’s a real number between 0 to 1 
Rough idea:

0= The prospect refuses long-term commitment. They want a one-off purchase, a pilot project, or a strict month-to-month subscription. This deal carries a high chunk risk after the initial period ends

0.5= The prospect has not ruled out a long-term deal, but it will depend entirely on pricing discounts, service-level guarantees, or product performance during a trial phase. This is an invitation for the sales representative to negotiate

1= The ideal enterprise client. They are looking for a strategic partner, stability, and locked-in pricing for the next 2 to 3 years. This deal represents high financial stability

P= It represents how difficult the procurement and approval process is. Higher complexity lowers the effective probability of budget conversion,  real number between 0.5 to 1, Rough idea:

1: The prospect can buy using a corporate credit card or a simple click-through agreement. No formal bidding, no complex legal reviews, and zero administrative overhead. The deal can close in days.

0.75: Requires passing an internal IT security review, signing a custom Master Services Agreement (MSA), and setting up a formal Purchase Order (PO) through their finance department. Expect a 1-to-3-month cycle.

0.5: The prospect forces us to go through a massive, competitive Request for proposal process. It involves strict compliance documentation, endless legal iterations, background checks, and multiple committee sign-offs. The sales cycle could take 6 to 12+ months, with a high risk of stalling out entirely.


Our composite finance score:
X1= 0.45*br + 0.17*C + 0.12*F1 + 0.12*F2 + 0.08*M+ 0.06*P


In [3]:
b= data[0]["b"]
d= data[0]["d"]
d= data[0]["d"]
br= min(b/d,1)
c= data[0]["c"]
t= data[0]["t"]
f1=1-(t/12)
f2= data[0]["f2"]
m= data[0]["m"]
p= data[0]["p"]

x1=0.45*br+0.17*c+0.12*f1+0.12*f2+0.08*m+0.06*p
print(x1)

0.8365


## Need, Problem and Product Fit

This parameter evaluates solution–prospect fit, how well our product matches the customer’s needs, urgency, and technical environment. Varaibles using are:  

R= Industry level match, it measures how closely the prospect’s industry aligns with industries where our company already has proven success, experience, or reference customers. [0,1]

S= It measures how directly our product solves the customer’s specific problem or intended use, to which extent our product is solving their problem statement. [0,1]

P= Problem statement’s idea, it measures how clearly the prospect can explain the problem they are trying to solve. [0,1] 

0: when he gives very vague idea of the problem statement
1: when he gives detailed properly documented problem statement

Cs= Current solution exists, it measures whether the customer already uses another method or product {0, 0.5, 1}

D= Dissatisfaction level with the current problem, it measures how unhappy the prospect is with what they currently use. [0,1]
0: fully satisfied, 1: deeply dissatisfied 

T= Represents the percentage of the prospect’s stated requirements our product can handle immediately without custom work. [0,1] 

C= It measures how much additional custom development or configuration is needed [0,1]

C2= Compliance fit, it measures how well our product meets the legal, industry, security, or regulatory requirements that the prospect must follow before they are allowed to buy or use your solution. In many industries, a customer may like the product and have budget—but still cannot purchase it unless it satisfies mandatory compliance standards {0, 0.5, 1}

0: your product doesn't meet their regulatory requirements, 0.5: partially meets, 1.0: fully compliant

U= Number of used cases has our client articulated, more use cases often increase strategic value, it's an integer.
Tracking the exact number of distinct use cases a prospect shares is one of the most reliable indicators of where they are in their buying journey. It acts as a direct proxy for buyer intent and deal maturity. [integer]

F_p= 0.25*R+ 0.3*S+ 0.2*P+ + 0.25*T

F_n= 1- 0.4*D*Cs- 0.25*C- 0.35*(1-C2) 

X2=0.45*F_p+ 0.4*F_n+ 0.15*min (U/6, 1)


In [4]:
r= data[1]["r"]
s= data[1]["s"]
p= data[1]["p"]
cs= data[1]["cs"]
d= data[1]["d"]
t= data[1]["t"]
c= data[1]["c"]
c2= data[1]["c2"]
u= data[1]["u"]
fp=0.25*r+0.3*s+ 0.2*p+0.25*t
fn=1-0.4*d*cs-0.25*c-0.35*(1-c2)
x2=0.45*fp+0.45*fn+0.15*min(u/6,1)
print(x2)

0.789625


## Authority and Decision Structure

This parameter tells basically who is involved in the deal, how much authority they have, and how aligned they are internally. Variables using:

R= Primary contact level role, it represents the organisational seniority of the main person we are interacting with. Higher seniority usually means stronger influence, better visibility into internal priorities, and faster access to decision-makers {0.2, 0.4, 0.6, 0.8, 1}

D= Decision involvement of our contact, it measures how directly the primary contact influences the final purchase decision {0, 0.5, 1}
    
A_c (Contact authority score)=R*D

N= Total number of stakeholders identified as part of the purchasing decision, [integer].

N’=min(N/5,1)

O= It measures whether multiple departments aligned on the need for the solution [0,1]

P= It Measures how directly our team can reach senior stakeholders [0,1]

X3= 0.3*A_c+ 0.3*N’ + 0.25*O +0.15*P

In [5]:
r= data[2]["r"]
d= data[2]["d"]
n= data[2]["n"]
o= data[2]["o"]
p= data[2]["p"]
n2=min(n/5,1)
a=r*d
x3=0.3*a+0.3*n2+0.25*o+0.15*p
print(x3)

0.7849999999999999


## Timeline, Urgency, and Buying Stage

This parameter measures timing and urgency—basically how soon the customer is likely to make a decision and how much momentum the deal currently has. Variables using are:

T= Represents the estimated number of days remaining before the prospect is expected to make a final purchase decision, [integer]

T1= Trigger event happened, it shows whether a specific event has created urgency or accelerated the need to buy. {0,1}

T2= Days until the trigger event deadline, [integer].  

Ep = Evaluation process, it measures how structured the customer’s buying/evaluation process is and checks whether there is a clear path toward selection [0,1]

Ns= Next step defined, it shows  whether there is a clearly agreed next action scheduled between both sides. {0,1}

Cp= Competing priorities, it measures how much attention the prospect is dividing between this deal and other business priorities and so captures the distraction risk. [0,1]

0: this is their top priority, 1 = many competing	

T’= 1.0 if T<=30
    
   = 1- (T-30)/150 if T>30 and T<=180
        
   = max (0, 0.1- (T-180)/1800) if T>180

U_trig= T1* max (0, 1- T2/180) 
    
P’=0.5*Ep+ 0.5*Ns
    
Cp’= 1-0.3*Cp
	
X4= Cp’*(0.5*T’ + 0.2*U_trig + 0.3*P’)


In [6]:
t= data[3]["t"]
t1= data[3]["t1"]
t2= data[3]["t2"]
ep= data[3]["ep"]
ns= data[3]["ns"]
cp= data[3]["cp"]
if t<=30:
    t3=1
elif t>30 and t<=180:
    t3=1-(t-30)/150
else:
    t3=max(0, 0.1-(t-180)/1800)
u=t1*max(0, 1- t2/180)
p2=0.5*ep+0.5*ns
cp2=1-0.3*cp

x4=cp2*(0.5*t3+0.2*u+0.3*p2)
print(x4)

0.8068666666666667


## Engagement Behaviour

This dimension measures engagement intensity and prospect activity—how actively the customer is interacting with our team and showing buying intent across different channels. Each variable captures a different part of that engagement. Variables used: 

N1= It is the total number of tracked email opens by the prospect across all sales communication. [integer]

N2= It measures the number of email responses sent by the prospect to your team. [integer]

N3= It represents the number of scheduled meetings that were actually completed with the prospect. [integer]

N4= It measures the number of meaningful phone or video calls completed with the prospect. [integer]

N5= It Measures the number of unique website sessions by the prospect. [integer]

N6= It Measures how many downloadable resources the prospect accesses. [integer]

N7= It measures how often the prospect visits pricing or package information. [integer]

N8= It measures formal requests made by the prospect to see or test the product. [integer]

N9= It measures engagement through social/community channels outside direct sales. [integer]

T= It measures how long it has been since the last real/meaningful interaction [integer]

V= Engagement velocity, it measures whether engagement is increasing or slowing over time. It is ratio of engagements in the last 14 days vs. prior 14 days. [float]

C= Channel diversity, it is number of distinct channels our prospect uses. (email, phone, web, social, in-person, etc.) [integer]

M= Negative signals, Counts interactions that reduce confidence or indicate resistance (unsubscribe, meeting cancellation, no-show, "not interested”, delayed response after commitment) [integer]



Er= 1*N1 + 3*N2 + 8*N3 +5*N4+ 2*N5+ 4*N6+ 6*N7+ 10*N8 +2*N9

En= 1/(1+e^(-0.08*(E_raw-25)), normalised between 0 and 1, 

R_decay(r)=e^(-0.033*T)

V= Ratio of engagements in the last 14 days vs. prior 14 days 

Vb=min (VB,2) *0.15 if VB>1 else it is 0
    
d=min(C/5,1) *0.1
    
n=max (0, 1-0.1*M)
    
X5=n(En*r+Vb+D_c)


In [7]:
import math

n1= data[4]["n1"]
n2= data[4]["n2"]
n3= data[4]["n3"]
n4= data[4]["n4"]
n5= data[4]["n5"]
n6= data[4]["n6"]
n7= data[4]["n7"]
n8= data[4]["n8"]
n9= data[4]["n9"]
t= data[4]["t"]
v= data[4]["v"]
c= data[4]["c"]
m= data[4]["m"]

er= n1+ 3*n2+ 8*n3+ 5*n4+2*n5+4*n6+6*n7+10*n8+2*n9
en=1/(1+math.exp(-0.08*(er-25)))
r=math.exp(-0.033*t)
if v>=1:
    vb=min(v,2)*0.15
else:
    vb=0
d=min(c/5,1)*0.1
n=max(0,1-0.1*m)
x5=n*(en*r+vb+d)
if x5>=1:
    x5=1
print(x5)

0.9886259568615763


## Company and Market Fit

This dimension measures Ideal Customer Profile (ICP) fit and business viability, how closely the prospect matches the kind of customer our company is best built to serve, and how practical it is to do business with them. Varaiables using are:

Seg= This variable measures how closely the prospect belongs to the specific customer segment your company targets most successfully. [0,1]

Emp= It measures how closely the company’s employee size matches your ideal customer size. [0,1]

Rev= Annual revenue relative to your ideal range [0,1]

Tech= It measures how well the prospect’s existing technology environment works with our product and checks compatibility with systems. [0,1]

Geo= Geographic alignment, it measures how well the prospect’s location fits your company’s active service regions. [0,1]

Gro= Company growth trajectory, it measures how the company is growing over time [0,1]

F= Financial health / credit risk, it measures how financially stable and low-risk the company is. [0,1]

D= Their readiness to adopt our type of technology, it measures how prepared the prospect is to adopt digital tools like ours. [0,1]

L= It measures how effectively you can work with the customer from a communication and business-culture perspective. [0,1]

X6= 0.20*seg+0.10*emp+0.10*rev+0.20*tech+0.05*geo+0.10*gro+0.10*F+0.1*D++0.05*L



In [8]:
seg= data[5]["seg"]
emp= data[5]["emp"]
rev= data[5]["rev"]
tech= data[5]["tech"]
geo= data[5]["geo"]
gro= data[5]["gro"]
f= data[5]["f"]
d= data[5]["d"]
l= data[5]["l"]
x6= 0.20*seg+0.10*emp+0.10*rev+0.20*tech+0.05*geo+0.10*gro+0.10*f+0.1*d+0.05*l
print(x6)

0.82


## Lead Source Quality

This parameter evaluates lead source quality and where the lead came from, how strong that source usually performs, and how much useful information we had when the lead entered our pipeline. Variables using are:

Q= Source channel quality, it measures the historical effectiveness of the channel through which the lead entered our pipeline. [0,1]

0.15: Purchased list / cold outbound- We bought their data or guessed their email. They have not shown any interest in our product, and we are starting from zero trust

0.25: Paid social ad- They clicked an ad on LinkedIn or Twitter. They fit our target demographic, but they were originally there to network or scroll, not to solve a business problem.

0.35: Paid search ad- They typed a specific problem into a search engine. This shows active intent, but they clicked a "Sponsored" link, which consumers naturally trust a bit less.

0.45: Content marketing- They are consuming our material. Trust is building, but they may just be looking for free advice rather than a paid tool.

0.5: Organic search- They searched a problem and clicked an organic, non-sponsored link to our site. This shows both active intent and a higher level of trust in authority than a paid ad.

0.55: Webinar attendee- They committed an hour of their schedule to listen to our experts. This shows serious interest in the topic.

0.6: Event / conference lead- They interacted with us in a professional, industry-specific setting, often face-to-face. Real-world interaction drastically reduces friction and builds immediate trust.

0.7: Free trial / freemium conversion- They are already inside our product. The intent is massive because they are actively testing to see if our solution fixes their problem. They just need to be convinced to pay for the premium version.

0.75: Partner referral- A trusted business partner vouched for us. The prospect transfers the trust they have in the partner directly to us.

0.85: Inbound RFP- A Request for Proposal means they have an active project, a timeline, and a budget. They are actively evaluating vendors to write a check.

0.9: Customer referral- An existing, happy customer told their peer to use our product. We don't need to sell them on our credibility; the customer already did it for us.

P= Campaign/asset quality, performance of the specific campaign relative to the best-performing campaign. [0,1]

R= How much data was captured at the point of entry? 0 = just an email, 1 = full profile with company, role, phone, and stated interest. [0,1]

S= {0.4, 1} 0.4 = outbound (we reached out), 1.0 = inbound (they came to us)

X7= 0.55*Q +0.15*P+ 0.10*R+0.20*S


In [9]:
q= data[6]["q"]
p= data[6]["p"]
r= data[6]["r"] 
s= data[6]["s"]
x7= 0.55*q+0.15*p+0.1*r+0.2*s
print(x7)

0.8025


## Competetive Landscape

This dimension evaluates competitive pressure and our position relative to alternatives—basically who else the customer is considering, how strong those competitors are, and how favorable your position is in comparison. Variables using are:

N= It is the total number of competing vendors or alternative solutions that the prospect is actively evaluating alongside our offering.. [integer] 

W= It measures our company’s historical success rate when competing against these same competitors in this specific customer segment. [0,1]

S= It measures how strongly the prospect is already tied to an existing vendor or current provider [0,1] 
 
0: no incumbent or weak incumbent, 1: deeply entrenched competitor with long contract 

D= It measures how clearly and convincingly you can explain why your solution is meaningfully different and better than alternatives for this prospect’s needs. [0,1] 

C= It measures how difficult and costly it would be for the prospect to switch from their current approach to your solution. [0,1] 

0: trivial to switch, 1: massive switching cost (migration, retraining, data transfer) 

K= It tells whether our company is the only vendor currently being considered. {0,1}, 1: If we are the sole vendor being considered 

n1=max(0, 1- N/6) 

cd= 1- 0.5*S- 0.5*C 

X8 = 1 						              if K=1 
   = 0.25*N’+ 0.3*W+ 0.2*cd+0.25*D    otherwise
       




In [10]:
n= data[7]["n"]
w= data[7]["w"]
s= data[7]["s"]
d= data[7]["d"]
c= data[7]["c"]
k= data[7]["k"]
n1=max(0,1-n/6)
cd=1-0.5*s-0.5*c
if k==1:
    x8=1
else:
    x8=0.25*n1+0.3*w+0.2*cd+0.25*d
print(x8)


0.635


## Relationship and Trust Equity 

This dimension measures relationship strength and trust between our organisation and the prospect— how familiar they are with us, how much confidence they have in our company, and whether past interactions make the deal easier or harder. Variables using are: 

Pb= It measures whether our organisation has had any previous business relationship with the prospect. {0, 0.5, 1} 0 = net new, 0.5 = lapsed customer or informal history, 1.0 = active or recent customer 

Rt= Relationship tenure, it is the duration of any prior relationship. [integer]

Nps= It measures the prospect’s overall sentiment toward our organisation. [-1,1], -1: detractor, 0: passive, 1: promoter. For new leads, default to 0 

Es= Executive sponsor relationship, it measures the strength of our relationship with a senior executive or decision-maker inside the prospect organisation. {0, 0.5, 1} 0 = no relationship, 0.5 = acquaintance, 1.0 = strong personal relationship with a decision-maker 

T= Trust indicator level, it measures observable signals that the prospect trusts our organisation enough to be transparent and collaborative- have they shared confidential information, introduced us to other stakeholders, given us early access to requirements. [0,1] 

Rc= reference customer, it measures whether wew can point to a successful customer that this prospect personally knows—or one highly relevant in their peer group.

N= Previous negative experience, it measures whether the prospect has had a bad past experience with our company—and how serious that experience was. {0, 0.5, 1} 

R_ten= Pb* min(ln(1+Rt)/ln (37), 1), it will be diminishing returns past 36 months 

R_trust=0.3*Es+ 0.3*T+ 0.2* Rc+ 0.2*max(Nps,0) 

R_neg=1-0.5*n 

X9=R_neg*( 0.4*R_ten+ 0.6* R_trust) 


In [11]:
pb= data[8]["pb"]
rt= data[8]["rt"]
nps= data[8]["nps"]
es= data[8]["es"]
t= data[8]["t"]
rc= data[8]["rc"]
n= data[8]["n"]
rten = pb* min(math.log(1+rt)/math.log(37),1)
rtrust = 0.3*es+0.3*t+0.2*rc+0.2*max(nps,0)
rneg= 1-0.5*n
x9= rneg*( 0.4*rten+ 0.6*rtrust)
print(x9)

0.6981706912645123


## Strategic and Lifetime Value 

This dimension measures long-term account value and strategic business impact—not just “Will this deal close?”, but “How valuable will this customer be over time?". Variables using are:

Ltv= Estimated lifetime value, this variable measures the total expected revenue our company will earn from the customer over the full duration of the relationship 

Cac= Customer acquisition cost, it measures the relationship between customer lifetime value and the cost required to acquire that customer. (total sales and marketing cost)/(number of new customers aquired)

Cac1= Customer acquisition cost ratio, it measures the total value a customer brings to our business over their entire lifespan compared to what it cost to get them through the door = Ltv/Cac 

Cs= It measures how much opportunity exists to sell additional products or services to this customer after the initial purchase. [0,1] 

B= Brand value of our client, it measures how valuable winning this account would be from a reputation and market credibility perspective. [0,1] 

M= It measures whether winning this customer helps your company enter or strengthen presence in a new market. {0, 0.5, 1} 

E= Expansion revenue probability, it measures the likelihood that the account will expand after the initial purchase. [0,1] 

C= Churn risk indicator, it measures the predicted risk that the customer may stop buying, cancel, or fail to renew in the future. [0,1] 

V_itv= 1/(1+e^(-1.5(Cac’-3))) 

V_strat= 0.3*B+ 0.4*M+ 0.3*E 

V_churn=1-0.5*C 

X10= V_churn (0.35*V_itv+ 0.25*Cs+ 0.4*V_strat) 


In [12]:
ltv=data[9]["ltv"]
cac=data[9]["cac"]
cac1= ltv/cac
cs= data[9]["cs"]
b= data[9]["b"]
m= data[9]["m"]
e= data[9]["e"]
c= data[9]["c"]
vitv= 1/(1+math.exp(-1.5*(cac1-3)))
vstrat=0.3*b+0.4*m+0.3*e
vchurn= 1-0.5*c
x10= vchurn*(0.35*vitv+0.25*cs+0.4*vstrat)
print(x10)

0.6022982208633029


## Now let’s consider some binary hard knockout cases:

G1: This gate checks whether the prospect has any realistic financial ability to buy. The customer may genuinely like the product and even have urgency—but if they have no budget now and no credible path to secure budget soon, the opportunity is not commercially viable.

G2: This gate ensures legal and ethical communication compliance. Contact has opted out, unsubscribed, or sent cease-and-desist.

G3: This gate protects pipeline quality. It checks whether the lead should actually be treated as a new sales opportunity or not and so checks whether our lead is a duplicate record or already an active customer (route to account management instead)

G4: Company or individual is on a sanctions list, trade restriction, or compliance blacklist. Are we legally allowed to do business with this organisation or person.

G5: Company is in a country/region you legally cannot sell to or support

G6: Company operates in an industry your organisation has a formal policy against serving,that is whether prospect belongs to an industry your company formally refuses to serve.

G7: After technical assessment, whether our product fundamentally can serve this prospect's core need — no roadmap path

They are all either 0 when the above cases happen, instantly closing our deal or 1 if their respective cases don’t happen.

G = G1 * G2 * G3 * G4 * G5 * G6 * G7


In [13]:
g1=[]
cases=["Confirmed budget = 0 AND no identified funding path or budget cycle within 18 months",
       "Contact has opted out, unsubscribed, or sent cease-and-desist",
       " Lead is a duplicate record or already an active customer, route to account management instead",
       "Company or individual is on a sanctions list, trade restriction, or compliance blacklist",
       "Company is in a country/region you legally cannot sell to or support",
       "Company operates in an industry your organisation has a formal policy against serving",
       "After technical assessment, your product fundamentally cannot serve this prospect's core need — no roadmap path"]
for i in range(len(gates)):
    h=gates[i]
    if h == 't':
        g1.append(0)
    else:
        g1.append(1)
g=1
for h in g1:
    g=g*h
print(g)

1


## Now we will go through some extreme cases that would directly influence the deal:

M1: Measures how reliable your deal data is—whether key details like budget, company size, contacts, and requirements are confirmed rather than estimated or missing
=0.5+0.5⋅data_completeness


M2: Checks whether your sales team has enough bandwidth and the right representative available to actively manage and close this deal.

= 1.0 if yes, 0.7 if stretched, 0.5 if no available rep in region/segment


M3: Evaluates how difficult the customer’s legal or contract conditions are, such as strict liability clauses, special IP terms, or non-standard agreements.

= 1.0 if standard terms, down to 0.6 for extreme requirements


M4: Measures the chance of payment issues due to currency fluctuations, long payment cycles, or customers located in higher-risk markets.

= 1.0 if low risk, down to 0.7


M5: Looks at how difficult the deal is to execute—for example multiple locations, language requirements, technical customization, or complex approval processes.

= 1.0 if simple, down to 0.6 for very complex deals


M=M1 * M2 * M3 * M4 * M5


In [14]:
a=[]
for i in range(len(modifiers)):
    h=modifiers[i]
    a.append(h)
m=1
for i in range(5):
    m=m*a[i]
print(m)


0.9


## Calculating Final Score

Now we will assign weights to our perimeters, making a final score, including all the hard knockout cases and things directly influencing the deal.

Let’s assign weights now: w1, w2, w3, w4, w5, w6, w7, w8, w9, w10

Such that: w1+w2+w3+w4+w5+w6+w7+w8+w9+w10=1

And our final score will be: 

S=100 * m * g * (w1 * x1 + w2 * x2 + w3 * x3 + w4 * x4 + w5 * x5 + w6 * x6 + w7 * x7 + w8 * x8 + w9 * x9 + w10 * x10)


In [15]:
x=[x1,x2,x3,x4,x5,x6,x7,x8,x9,x10]
print(x)
wx=0
for i in range(10):
    wx+=weights[i]*x[i]    
Score=100*m*g*wx
print(Score)

[0.8365, 0.789625, 0.7849999999999999, 0.8068666666666667, 0.9886259568615763, 0.82, 0.8025, 0.635, 0.6981706912645123, 0.6022982208633029]
71.92876723623414
